In [1]:
# Imports
from lxml import etree
import sumolib
import pandas as pd

In [ ]:
NETWORK = "../../sumo/berlin.net.xml"
ADDITIONAL = "../data_basis/termination_points.add.xml"

# Load network
net = sumolib.net.readNet(NETWORK)

# Parse charging stations
tree = etree.parse(ADDITIONAL)
root = tree.getroot()

In [ ]:
stations = []

for cs in root.findall("chargingStation"):
    lane = cs.attrib["lane"]
    edge = lane.rsplit("_", 1)[0]
    stations.append({
        "id": cs.attrib["id"],
        "name": cs.attrib["name"],
        "edge": edge,
    })

for skip_idx, output_file in [
    (0, "../preprocessing_output/deadhead_time_cicerostrasse.txt"),
    (1, "../preprocessing_output/deadhead_time_muellerstrasse.txt"),
]:
    selected = stations[:skip_idx] + stations[skip_idx + 1:]
    selected[0]["id"] = "1"

    time_rows = []
    routes = []

    for origin in selected:
        from_edge = net.getEdge(origin["edge"])

        for dest in selected:
            to_edge = net.getEdge(dest["edge"])
            edges, cost = net.getFastestPath(from_edge, to_edge)
            
            edge_ids = [edge.getID() for edge in edges]
            time_rows.append({
                "FromStopID": origin["id"],
                "ToStopID": dest["id"],
                "RunTime": round(cost),
            })
            routes.append({
                "FromStopID": origin["id"],
                "ToStopID": dest["id"],
                "Edges": " ".join(edge_ids),
            })

    pd.DataFrame(time_rows).to_csv(output_file, sep=";", index=False)

In [ ]:
root = etree.Element(
    "routes",
    nsmap={
        "xsi": "http://www.w3.org/2001/XMLSchema-instance",
    },
)

root.set(
    "{http://www.w3.org/2001/XMLSchema-instance}noNamespaceSchemaLocation",
    "http://sumo.dlr.de/xsd/routes_file.xsd",
)

# Vehicle type
etree.SubElement(
    root,
    "vType",
    id="bus",
    vClass="bus",
)

# Routes
for route in routes:
    etree.SubElement(
        root,
        "route",
        id=f"{route['FromStopID']}_{route['ToStopID']}",
        color="240,215,34",
        edges=route["Edges"],
    )

# Write XML
tree = etree.ElementTree(root)
tree.write(
    "../preprocessing_output/deadheads_cicero_mueller.rou.xml",
    encoding="utf-8",
    xml_declaration=True,
    pretty_print=True,
)